In [1]:
import xgboost as xgb

model = xgb.Booster()
model.load_model("citysense_final_model.json")

print("Model loaded successfully")
print("Trees:", model.num_boosted_rounds())

Model loaded successfully
Trees: 600


In [2]:
import pandas as pd

df = pd.read_parquet(
    "data/yearly/citysense_2025.parquet"
)

print(df.columns.tolist())
print(df[[
    "grid_id",
    "cmplnt_fr_dt",
    "hour",
    "crime_count",
    "historical_grid_crime_count",
    "historical_grid_hour_crime_count",
    "historical_grid_day_crime_count",
    "historical_grid_time_period_crime_count",
    "historical_grid_weekend_crime_count",
    "lat_grid",
    "lon_grid"
]].head())

['grid_id', 'cmplnt_fr_dt', 'hour', 'crime_count', 'historical_grid_crime_count', 'historical_grid_hour_crime_count', 'day_of_week', 'historical_grid_day_crime_count', 'time_period', 'historical_grid_time_period_crime_count', 'is_weekend', 'historical_grid_weekend_crime_count', 'year', 'month', 'lat_grid', 'lon_grid']
        grid_id cmplnt_fr_dt  hour  crime_count  historical_grid_crime_count  \
0  40.49_-74.24   2025-01-01     0            0                           15   
1  40.49_-74.24   2025-01-01     1            0                           15   
2  40.49_-74.24   2025-01-01     2            0                           15   
3  40.49_-74.24   2025-01-01     3            0                           15   
4  40.49_-74.24   2025-01-01     4            0                           15   

   historical_grid_hour_crime_count  historical_grid_day_crime_count  \
0                                 2                                1   
1                                 0                    

In [3]:
print("Unique grids:", df["grid_id"].nunique())
print("Grid latitude range:", df["lat_grid"].min(), "to", df["lat_grid"].max())
print("Grid longitude range:", df["lon_grid"].min(), "to", df["lon_grid"].max())

print("\nTime periods:")
print(df["time_period"].cat.categories)

Unique grids: 946
Grid latitude range: 40.49 to 40.91
Grid longitude range: -74.26 to -73.71

Time periods:
Index(['Night', 'Morning', 'Afternoon', 'Evening'], dtype='str')


In [4]:
print(
    df[["grid_id", "lat_grid", "lon_grid"]]
    .drop_duplicates()
    .head(20)
    .to_string(index=False)
)

     grid_id  lat_grid   lon_grid
40.49_-74.24 40.490002 -74.239998
40.49_-74.25 40.490002 -74.250000
40.51_-74.19 40.509998 -74.190002
 40.51_-74.2 40.509998 -74.199997
40.51_-74.21 40.509998 -74.209999
40.51_-74.22 40.509998 -74.220001
40.51_-74.23 40.509998 -74.230003
40.51_-74.24 40.509998 -74.239998
40.51_-74.25 40.509998 -74.250000
40.51_-74.26 40.509998 -74.260002
40.52_-74.16 40.520000 -74.160004
40.52_-74.17 40.520000 -74.169998
40.52_-74.18 40.520000 -74.180000
40.52_-74.19 40.520000 -74.190002
 40.52_-74.2 40.520000 -74.199997
40.52_-74.21 40.520000 -74.209999
40.52_-74.22 40.520000 -74.220001
40.52_-74.23 40.520000 -74.230003
40.52_-74.24 40.520000 -74.239998
40.52_-74.25 40.520000 -74.250000


In [5]:
lat_values = sorted(df["lat_grid"].unique())
lon_values = sorted(df["lon_grid"].unique())

print("Latitude spacing:")
print(pd.Series(lat_values).diff().dropna().value_counts().head())

print("\nLongitude spacing:")
print(pd.Series(lon_values).diff().dropna().value_counts().head())

Latitude spacing:
0.009998    24
0.010002    18
Name: count, dtype: int64

Longitude spacing:
0.010002    40
0.009995    15
Name: count, dtype: int64


In [6]:
grid_lookup = (
    df[["grid_id", "lat_grid", "lon_grid"]]
    .drop_duplicates("grid_id")
    .reset_index(drop=True)
)

print("Grid cells:", len(grid_lookup))
print(grid_lookup.head())

Grid cells: 946
        grid_id   lat_grid   lon_grid
0  40.49_-74.24  40.490002 -74.239998
1  40.49_-74.25  40.490002 -74.250000
2  40.51_-74.19  40.509998 -74.190002
3   40.51_-74.2  40.509998 -74.199997
4  40.51_-74.21  40.509998 -74.209999


In [7]:
import numpy as np

def get_nearest_grid(latitude, longitude):
    distances = (
        (grid_lookup["lat_grid"] - latitude) ** 2
        + (grid_lookup["lon_grid"] - longitude) ** 2
    )

    idx = distances.idxmin()
    row = grid_lookup.loc[idx]

    return row["grid_id"], row["lat_grid"], row["lon_grid"]

In [8]:
grid_id, lat_grid, lon_grid = get_nearest_grid(
    40.7128,
    -74.0060
)

print("Grid ID:", grid_id)
print("Grid center:", lat_grid, lon_grid)

Grid ID: 40.71_-74.01
Grid center: 40.71 -74.01


In [9]:
grid_id = "40.49_-74.24"

print(
    "2025 historical:",
    df.loc[df["grid_id"] == grid_id, "historical_grid_crime_count"].iloc[0]
)

2025 historical: 15


In [10]:
grid_id = "40.49_-74.24"

total = 0

for year in range(2020, 2025):
    temp = pd.read_parquet(
        f"data/yearly/citysense_{year}.parquet",
        columns=["grid_id", "crime_count"]
    )

    year_total = temp.loc[
        temp["grid_id"] == grid_id,
        "crime_count"
    ].sum()

    print(year, year_total)
    total += year_total

print("2020-2024 total:", total)

2020 2
2021 2
2022 4
2023 3
2024 4
2020-2024 total: 15


In [11]:
import pandas as pd

history = []

for year in range(2020, 2026):
    temp = pd.read_parquet(
        f"data/yearly/citysense_{year}.parquet",
        columns=[
            "grid_id",
            "hour",
            "crime_count",
            "day_of_week",
            "time_period",
            "is_weekend"
        ]
    )
    history.append(temp)

history = pd.concat(history, ignore_index=True)

print("Rows:", len(history))

Rows: 49767168


In [12]:
grid_history = (
    history.groupby("grid_id")["crime_count"]
    .sum()
    .rename("historical_grid_crime_count")
)

grid_hour_history = (
    history.groupby(["grid_id", "hour"])["crime_count"]
    .sum()
    .rename("historical_grid_hour_crime_count")
)

grid_day_history = (
    history.groupby(["grid_id", "day_of_week"])["crime_count"]
    .sum()
    .rename("historical_grid_day_crime_count")
)

grid_time_history = (
    history.groupby(["grid_id", "time_period"])["crime_count"]
    .sum()
    .rename("historical_grid_time_period_crime_count")
)

grid_weekend_history = (
    history.groupby(["grid_id", "is_weekend"])["crime_count"]
    .sum()
    .rename("historical_grid_weekend_crime_count")
)

In [13]:
grid_id = "40.49_-74.24"

print(
    "Grid history:",
    grid_history.loc[grid_id]
)

Grid history: 18


In [14]:
def get_historical_features(grid_id, date, hour):
    timestamp = pd.Timestamp(date)

    day_of_week = timestamp.dayofweek
    time_period = (
        "Night" if hour < 6 else
        "Morning" if hour < 12 else
        "Afternoon" if hour < 17 else
        "Evening"
    )
    is_weekend = int(day_of_week >= 5)

    return {
        "historical_grid_crime_count":
            grid_history.get(grid_id, 0),

        "historical_grid_hour_crime_count":
            grid_hour_history.get((grid_id, hour), 0),

        "historical_grid_day_crime_count":
            grid_day_history.get((grid_id, day_of_week), 0),

        "historical_grid_time_period_crime_count":
            grid_time_history.get((grid_id, time_period), 0),

        "historical_grid_weekend_crime_count":
            grid_weekend_history.get((grid_id, is_weekend), 0)
    }

In [15]:
features = get_historical_features(
    "40.49_-74.24",
    "2026-09-20",
    20
)

print(features)

{'historical_grid_crime_count': np.int32(18), 'historical_grid_hour_crime_count': np.int32(0), 'historical_grid_day_crime_count': np.int32(2), 'historical_grid_time_period_crime_count': np.int32(4), 'historical_grid_weekend_crime_count': np.int32(4)}


In [16]:
print(
    df[["hour", "time_period"]]
    .drop_duplicates()
    .sort_values("hour")
    .to_string(index=False)
)

 hour time_period
    0       Night
    1       Night
    2       Night
    3       Night
    4       Night
    5     Morning
    6     Morning
    7     Morning
    8     Morning
    9     Morning
   10     Morning
   11     Morning
   12   Afternoon
   13   Afternoon
   14   Afternoon
   15   Afternoon
   16   Afternoon
   17   Afternoon
   18     Evening
   19     Evening
   20     Evening
   21     Evening
   22     Evening
   23     Evening


In [17]:
def create_inference_row(latitude, longitude, date, hour):
    timestamp = pd.Timestamp(date)

    grid_id, lat_grid, lon_grid = get_nearest_grid(
        latitude,
        longitude
    )

    day_of_week = timestamp.dayofweek

    if hour < 5:
        time_period = "Night"
    elif hour < 12:
        time_period = "Morning"
    elif hour < 18:
        time_period = "Afternoon"
    else:
        time_period = "Evening"

    is_weekend = int(day_of_week >= 5)

    historical = get_historical_features(
        grid_id,
        date,
        hour
    )

    return pd.DataFrame([{
        "grid_id": grid_id,
        "hour": hour,
        "historical_grid_crime_count":
            historical["historical_grid_crime_count"],
        "historical_grid_hour_crime_count":
            historical["historical_grid_hour_crime_count"],
        "day_of_week": day_of_week,
        "time_period": time_period,
        "historical_grid_day_crime_count":
            historical["historical_grid_day_crime_count"],
        "historical_grid_time_period_crime_count":
            historical["historical_grid_time_period_crime_count"],
        "is_weekend": is_weekend,
        "historical_grid_weekend_crime_count":
            historical["historical_grid_weekend_crime_count"],
        "year": timestamp.year,
        "month": timestamp.month,
        "lat_grid": lat_grid,
        "lon_grid": lon_grid
    }])

In [18]:
X = create_inference_row(
    40.7128,
    -74.0060,
    "2026-09-20",
    20
)

print(X.T)

                                                    0
grid_id                                  40.71_-74.01
hour                                               20
historical_grid_crime_count                     15141
historical_grid_hour_crime_count                  524
day_of_week                                         6
time_period                                   Evening
historical_grid_day_crime_count                  1566
historical_grid_time_period_crime_count          3044
is_weekend                                          1
historical_grid_weekend_crime_count              3442
year                                             2026
month                                               9
lat_grid                                    40.709999
lon_grid                                   -74.010002


In [20]:
GRID_CATEGORIES = pd.read_parquet(
    "data/yearly/citysense_2025.parquet",
    columns=["grid_id"]
)["grid_id"].cat.categories

TIME_CATEGORIES = pd.read_parquet(
    "data/yearly/citysense_2025.parquet",
    columns=["time_period"]
)["time_period"].cat.categories

In [21]:
X["grid_id"] = pd.Categorical(
    X["grid_id"],
    categories=GRID_CATEGORIES
)

X["time_period"] = pd.Categorical(
    X["time_period"],
    categories=TIME_CATEGORIES
)

dtest = xgb.DMatrix(
    X,
    enable_categorical=True
)

probability = model.predict(dtest)[0]

print("Crime probability:", probability)

Crime probability: 0.2221943
